# 🚨 Sesiunea 20 · Anomalii și serii de timp

Azi lucrăm cu **date reale**, pas cu pas:

1. **Fraude cu cardul**: 284.807 tranzacții reale. Care sunt suspecte?
2. **Taxiurile din New York**: câți pasageri vor fi zilele următoare? Ce zile au fost ciudate?
3. **Mini proiect**: un senzor de pe o mașină industrială. Găsești singur defectele?

Ideea care leagă totul: **o anomalie = ceva foarte diferit de normal**.

## Pregătire (rulează o dată)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

O funcție ajutătoare: ne spune **în cuvinte** cât de bine am prins fraudele. Nu trebuie să înțelegi codul ei.

In [ ]:
def arata_rezultate(semnalate):
    frauda = tranzactii["Class"] == 1
    prinse = (semnalate & frauda).sum()
    print("Am semnalat", semnalate.sum(), "tranzacții. Dintre ele,", prinse, "chiar sunt fraude.")
    print("Am prins", prinse, "din", frauda.sum(), "fraude.")

# Partea I · Fraude cu cardul 💳
*(după slide-ul „Datele: fraude cu cardul”)*

Tranzacții reale făcute cu carduri europene, în **2 zile din septembrie 2013**. Setul e public și foarte folosit pentru a învăța detecția fraudelor.

## 1 · Încărcăm datele

In [ ]:
tranzactii = pd.read_csv("https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv")

Câte rânduri și câte coloane avem?

In [ ]:
tranzactii.shape

In [ ]:
tranzactii.head()

Ce înseamnă coloanele:
- **Time**: câte secunde au trecut de la prima tranzacție
- **V1 … V28**: informații despre tranzacție, **ascunse de bancă** (transformate, ca să nu se vadă date personale)
- **Amount**: suma, în euro
- **Class**: răspunsul corect. **1 = fraudă**, 0 = normal

Câte fraude sunt?

In [ ]:
tranzactii["Class"].value_counts()

**🤔 Întrebare** · Un „model” care spune mereu „nu e fraudă” ar avea 99,83% acuratețe. E un model bun?

> ✍️ *Răspunsul tău:* ....

Sunt fraudele sume mari? Ne uităm la **suma tipică** (mediana) pentru fiecare clasă.

In [ ]:
tranzactii.groupby("Class")["Amount"].median()

**🤔 Întrebare** · Te așteptai la asta? De ce crezi că multe fraude sunt sume mici?

> ✍️ *Răspunsul tău:* ....

## 2 · O regulă simplă: sumele mari 💰
*(după slide-ul „O regulă simplă”)*

**Exercițiul 2.1 · Prima regulă**

Completează regula: semnalăm tranzacțiile cu `Amount` **mai mare de 1000**. Apoi încearcă 500 și 100. Ce se întâmplă?

> ✍️ *Răspunsul tău:* ....

In [ ]:
semnalate = # ...   (Amount mai mare de 1000)

In [ ]:
arata_rezultate(semnalate)

## 3 · Isolation Forest: toate coloanele deodată 🌲
*(după slide-urile „Isolation Forest”)*

Isolation Forest caută singur tranzacțiile care **nu seamănă** cu celelalte. Nu vede coloana `Class`: nu știe care sunt fraude, doar ce e „ciudat”.

**Pasul 1:** alegem coloanele. Toate, în afară de `Time` și de `Class` (răspunsul).

In [ ]:
coloane = tranzactii.columns.drop(["Time", "Class"])
coloane

**Pasul 2:** creăm modelul. `contamination` = ce procent din tranzacții credem că sunt ciudate. Știm că fraudele sunt ~0,17%, deci punem **0.002**.

In [ ]:
model = IsolationForest(contamination=0.002, random_state=0)

**Pasul 3:** modelul învață cum arată tranzacțiile (durează câteva secunde).

In [ ]:
model.fit(tranzactii[coloane])

**Pasul 4:** îl întrebăm, pentru fiecare tranzacție: e ciudată? Răspunsul e **-1** (ciudat) sau **1** (normal).

In [ ]:
rezultat = model.predict(tranzactii[coloane])
tranzactii["ciudata"] = rezultat == -1

**Pasul 5:** verificăm cu răspunsul corect.

In [ ]:
arata_rezultate(tranzactii["ciudata"])

**🤔 Întrebare** · Compară cu regula de la secțiunea 2. Cine a prins mai multe fraude? De ce?

> ✍️ *Răspunsul tău:* ....

**Exercițiul 3.1 · Cât de „paranoic” să fie modelul?**

Schimbă `contamination` (încearcă 0.001, 0.005, 0.01) și rulează. Ești banca: ce valoare alegi? Ce costă mai mult: o fraudă ratată sau un client cu cardul blocat degeaba?

> ✍️ *Răspunsul tău:* ....

In [ ]:
model = IsolationForest(contamination= # ... , random_state=0)
model.fit(tranzactii[coloane])
arata_rezultate(model.predict(tranzactii[coloane]) == -1)

# Partea a II-a · Taxiurile din New York 🚕
*(după slide-ul „Datele: taxiuri în New York”)*

Câți pasageri au luat taxiul în New York, **la fiecare 30 de minute**, din iulie 2014 până în ianuarie 2015. Setul face parte din **NAB** (Numenta Anomaly Benchmark), o colecție publică de date pentru detecția anomaliilor.

## 4 · Încărcăm și privim datele

In [ ]:
taxi = pd.read_csv("https://raw.githubusercontent.com/numenta/NAB/master/data/realKnownCause/nyc_taxi.csv")
taxi.head()

Spunem Python-ului că prima coloană e **timp**, și o folosim ca index.

In [ ]:
taxi["timestamp"] = pd.to_datetime(taxi["timestamp"])
taxi = taxi.set_index("timestamp")

O singură zi, ca să vedem ritmul orelor:

In [ ]:
taxi.loc["2014-07-08"].plot(title="Pasageri la fiecare 30 de minute, o zi de marți")
plt.show()

Datele la 30 de minute sunt prea multe. Le **adunăm pe zile**.

In [ ]:
pe_zi = taxi["value"].resample("D").sum()
pe_zi.plot(title="Pasageri pe zi")
plt.show()

**🤔 Întrebare** · Ce tipare vezi în grafic? Ce zile îți sar în ochi?

> ✍️ *Răspunsul tău:* ....

Media pasagerilor pe fiecare zi a săptămânii:

In [ ]:
pe_zi.groupby(pe_zi.index.day_name()).mean().sort_values().plot.bar(rot=45)
plt.show()

**🤔 Întrebare** · Care e cea mai aglomerată zi? Care e cea mai liniștită? Ce înseamnă asta pentru o prognoză?

> ✍️ *Răspunsul tău:* ....

## 5 · Prognoza: ghicim ultimele 4 săptămâni 🔮
*(după slide-urile „Trecutul antrenează, viitorul testează” și „Trei prognoze simple”)*

Ascundem ultimele **28 de zile** și ne prefacem că sunt viitorul.

In [ ]:
real = pe_zi[-28:]
real.head()

Trei prognoze simple:
- **Naiv:** mâine = azi → `pe_zi.shift(1)` (valoarea de ieri)
- **Sezonier:** ca acum o săptămână → `pe_zi.shift(7)`
- **Tipic:** valoarea tipică din ultimele 4 săptămâni, pentru aceeași zi

In [ ]:
naiv = pe_zi.shift(1)[-28:]

**Exercițiul 5.1 · Prognoza sezonieră**

Completează: valoarea de **acum 7 zile**, pentru ultimele 28 de zile.

> ✍️ *Răspunsul tău:* ....

In [ ]:
sezonier = # ...   (valoarea de acum 7 zile, ultimele 28 de zile)

In [ ]:
ultimele_4 = pd.concat([pe_zi.shift(7), pe_zi.shift(14), pe_zi.shift(21), pe_zi.shift(28)], axis=1)
tipic = ultimele_4.median(axis=1)[-28:]

Cu câți pasageri greșim, **în medie, pe zi**? (diferența dintre real și prognoză, fără semn, apoi media)

In [ ]:
print("naiv:    ", (real - naiv).abs().mean().round())
print("sezonier:", (real - sezonier).abs().mean().round())
print("tipic:   ", (real - tipic).abs().mean().round())

In [ ]:
real.plot(label="real", color="black")
naiv.plot(label="naiv", linestyle=":")
tipic.plot(label="tipic")
plt.legend()
plt.show()

**🤔 Întrebare** · Care prognoză e cea mai slabă? Și unde greșesc **toate** foarte mult?

> ✍️ *Răspunsul tău:* ....

## 6 · Ce zile au fost ciudate? 🕵️
*(după slide-ul „Zi ciudată = departe de ce era tipic”)*

Pentru **fiecare zi**: ce era tipic? Cu cât la sută e realul peste sau sub?

In [ ]:
tipic_tot = ultimele_4.median(axis=1, skipna=False)
diferenta = (pe_zi - tipic_tot) / tipic_tot

Păstrăm zilele cu peste **25%** diferență:

In [ ]:
PRAG = 0.25        # ✏️ încearcă și 0.15, 0.35
ciudate = diferenta[diferenta.abs() > PRAG]
ciudate.round(2)

In [ ]:
pe_zi.plot(color="gray")
plt.scatter(ciudate.index, pe_zi[ciudate.index], color="red", zorder=3)
plt.show()

**Exercițiul 6.1 · Detectivul**

Pentru fiecare zi găsită, ghicește ce s-a întâmplat (uită-te la dată!). Sunt toate probleme?

> ✍️ *Răspunsul tău:* ....

# 🏠 Mini proiect: senzorul unei mașini industriale
*(după slide-ul „Mini proiect”)*

Un senzor măsoară **temperatura unei mașini industriale mari**, la fiecare 5 minute, din decembrie 2013 până în februarie 2014. În perioada asta mașina s-a **oprit sau s-a defectat de 4 ori**. Datele sunt tot din NAB.

**Misiunea ta:** găsește zilele cu probleme, folosind ce ai învățat azi. Fiecare pas are un ✏️ de completat. Indiciile sunt în secțiunile 4 și 6.

**Pasul 1 · Încarcă datele** (URL-ul e dat) și afișează primele rânduri.

In [ ]:
URL = "https://raw.githubusercontent.com/numenta/NAB/master/data/realKnownCause/machine_temperature_system_failure.csv"
masina = # ...   (citește CSV-ul de la URL)
masina.head()

**Pasul 2 · Pregătește timpul:** transformă coloana `timestamp` în dată și pune-o ca index.

In [ ]:
masina["timestamp"] = # ...
masina = # ...

**Pasul 3 · Fă graficul** coloanei `value`. Vezi deja ceva ciudat?

In [ ]:
# ✏️ graficul coloanei "value"

**Pasul 4 · Media pe zi.** Atenție: la taxiuri am **adunat** pasagerii (`sum`). La temperatură, adunarea n-are sens: facem **media** (`mean`).

In [ ]:
pe_zi_m = # ...   (media pe fiecare zi)
pe_zi_m.plot()
plt.show()

**Pasul 5 · Ce era tipic?** O mașină nu are weekend: merge la fel în fiecare zi. Așa că „tipic” = **mediana ultimelor 7 zile** (fără ziua de azi). Codul e gata, citește-l.

In [ ]:
tipic_m = pe_zi_m.rolling(7).median().shift(1)

**Pasul 6 · Diferența în procente**, ca la taxiuri.

In [ ]:
diferenta_m = # ...   (real minus tipic, împărțit la tipic)

**Pasul 7 · Zilele cu probleme.** La o mașină ne îngrijorează când temperatura **scade** brusc (s-a oprit sau s-a defectat). Păstrează zilele cu o scădere mai mare de 20%.

In [ ]:
probleme = # ...   (indiciu: diferenta_m < -0.20)
probleme.round(2)

**Pasul 8 · Verifică.** Cercetătorii NAB au marcat aceste 4 perioade cu probleme reale:

In [ ]:
perioade_reale = ["10–11 decembrie 2013", "16–17 decembrie 2013", "28 ianuarie 2014", "7–9 februarie 2014"]
perioade_reale

**Pasul 9 · Răspunde pe scurt** (2–3 rânduri fiecare):

1. Câte din cele 4 perioade ai găsit?
2. Ce se întâmplă dacă pui pragul la 10%? Dar la 40%?
3. De ce am folosit aici „ultimele 7 zile” și nu „aceeași zi de acum o săptămână”, ca la taxiuri?
4. Dacă ai fi inginerul fabricii, ce ai face cu o alertă ca asta?

**✍️ Răspunsurile tale la pasul 9**

> ✍️ *Răspunsul tău:* ....

## Recap 🎓

**Anomalie = ceva foarte diferit de normal. Ca s-o găsești, întâi afli cum arată normalul.**

- **Cazuri rare:** acuratețea minte. Întrebăm: cât din ce semnalăm e fraudă? Câte fraude prindem?
- **Regula simplă** vede o singură coloană; **Isolation Forest** le vede pe toate, fără etichete.
- **Serie de timp:** ritm (zi, săptămână) + evenimente speciale.
- **Prognoză:** testăm pe ultimele săptămâni, comparăm cu prognoze simple.
- **Zi ciudată:** foarte departe de tipic. **Anomalie nu înseamnă problemă**: decide un om.